In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from cryojax.dataset import RelionParticleParameterFile, RelionParticleStackDataset
from cryojax.ndimage.transforms import LowpassFilter

import cryojax_eo as cxeo

# Let's load the config file

In [ ]:
config = cxeo.load_config("./config_data_simulation.yaml", config_mode="data simulation")

config.noise_snr

The image generation can be run from the command line as:

`simulate_data --config config_data_simulation.yaml`

if this fails, make sure your virtual environment is activated. This command simply runs the following function

In [ ]:
cxeo.dataset.simulate_relion_dataset(config)

# Visualize the images!

In [ ]:
stack_dataset = RelionParticleStackDataset(
    RelionParticleParameterFile(
        path_to_starfile=config.path_to_starfile,
        mode="r",
        loads_envelope=False,
    ),
    path_to_relion_project=config.path_to_relion_project,
    mode="r",
    loads_parameters=True,
)

In [ ]:
lowpass_filter = LowpassFilter(
    stack_dataset[0]["parameters"]["image_config"].frequency_grid_in_pixels,
    frequency_cutoff_fraction=0.7,
)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 10))

images = stack_dataset[0:4]["images"]
# images = irfftn(lowpass_filter(rfftn(images)))

for i in range(4):
    ax.flatten()[i].imshow(images[i], cmap="gray")

## Metadata

Information about the ensemble and other parameters is saved to a metadata file

In [ ]:
metadata = np.load("tutorial_data/metadata.npz")

metadata.files

In [ ]:
metadata["ensemble_indices_per_image"]

In [ ]:
weight_0 = np.isclose(metadata["ensemble_indices_per_image"], 0).mean()
weight_1 = np.isclose(metadata["ensemble_indices_per_image"], 1).mean()

weight_0, weight_1